# Multi-candidate attribution — the interface (D207)

**What this is.** One Run All produces the single attribution number `N` for the
30–60° DB-turn band, with its composition printed underneath and the identified
set beside the frozen estimate. It is the spec's §0 deliverable
(`notes/multi-candidate-spec.md`), assembled from four slots:

```
N = N_frozen + N_geo(5) + N_geo(8) + N_routeA
```

**What it is not.** The dataset records no opponent identifier. `N_frozen` is the
frozen elimination match and is the only slot labelled `RULE`; every other slot is
an **INFERENCE** and no output here says which man was covered. Nothing here enters
`TASK.md`, the band table, the four figures or the report draft.

**How to run it.** Edit the one config cell below, then Run All. With no tracking
present the geometric slots report `NOT EVALUABLE` and contribute zero; point
`TRACKING` at the full `game_tracking_20xx.csv` files and they are computed.
There is no CLI and nothing is installed.

In [ ]:
# ---- CONFIG — the only cell you edit -----------------------------------
# None means "the repo default". A path is used as given.
TURNS       = None   # -> data/interim/full_run_20261111/bdb27_full_db_turns.parquet
PLAYER_PLAY = None   # -> data/raw/player_play.csv
PLAYERS     = None   # -> data/raw/players.csv

# []  -> auto-discovery of data/raw/game_tracking*.csv, which deliberately
#        skips files with "sample" in the name: a truncation cannot answer the
#        distance questions and must not stand in for the full file.
# [...] -> full tracking file(s), used as given (that is how the sample file can
#        be pointed at deliberately, and how it then prints its truncation).
TRACKING    = []

RADII           = (5.0, 8.0)   # the D204 sensitivity radii, unchanged
SEED            = 20261117     # AC3's fixed seed
WRITE_ARTEFACTS = False        # True writes route_a/assembly CSVs next to the run
OUT_DIR         = "outputs/ambiguity_attribution_notebook"

In [ ]:
from pathlib import Path
import sys

import pandas as pd


def repo_root(start: Path) -> Path:
    """The nearest directory above ``start`` that carries ``src/bdb27``."""
    tried = []
    for p in [start, *start.parents]:
        tried.append(str(p))
        if (p / "src" / "bdb27" / "ambiguity_attribution.py").exists():
            return p
    raise FileNotFoundError(
        "cannot find the BDB27 repo root; looked in:\n  " + "\n  ".join(tried))


ROOT = repo_root(Path.cwd().resolve())
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

from bdb27 import ambiguity_attribution as aa
from bdb27 import outcome_test as ot

TURNS = Path(TURNS) if TURNS else ROOT / "data/interim/full_run_20261111/bdb27_full_db_turns.parquet"
PLAYER_PLAY = Path(PLAYER_PLAY) if PLAYER_PLAY else ROOT / "data/raw/player_play.csv"
PLAYERS = Path(PLAYERS) if PLAYERS else ROOT / "data/raw/players.csv"
OUT = Path(OUT_DIR) if Path(OUT_DIR).is_absolute() else ROOT / OUT_DIR

print("repo       ", ROOT)
print("turns      ", TURNS)
print("player_play", PLAYER_PLAY)
print("tracking   ", TRACKING if TRACKING else "(auto-discovery; sample-named files skipped)")

In [ ]:
con = aa.connect()
try:
    funnel = aa.funnel(TURNS, PLAYER_PLAY, con=con)
    single = aa.build_frame(TURNS, PLAYER_PLAY, status="single", con=con)
    multiple = aa.build_frame(TURNS, PLAYER_PLAY, status="multiple", con=con)
finally:
    con.close()

frame = aa.analysis_frame(single, multiple)
band_turns = int(funnel.loc[funnel["step"] == "band_turns", "n"].iloc[0])

print(funnel[["step", "n", "denominator", "matches"]].to_string(index=False))
print()
print(f"frozen frame  {len(single):,} turns  ·  ambiguous "
      f"{multiple[list(ot.TURN_KEY)].drop_duplicates().shape[0]:,} turns "
      f"({len(multiple):,} candidate rows)")

In [ ]:
# O2 — the identified set beside the frozen-only estimate (label set here only)
bounds = pd.DataFrame([aa.identified_set(frame, spec) for spec in aa.SPECS])
primary = aa.identified_set(frame, "primary")

print(bounds[["spec", "beta_frozen", "ci_lo", "ci_hi", "n_frozen", "frozen_clusters",
              "beta_min", "beta_max", "width", "ci_width", "label"]]
      .to_string(index=False, float_format=lambda v: f"{v:+.3f}"))
print()
print(f"Label, set by O2 alone: {primary['branch']}")
print(f"  frozen-only beta {primary['beta_frozen']:+.3f}  "
      f"95% CI [{primary['ci_lo']:+.3f}, {primary['ci_hi']:+.3f}]  "
      f"n={primary['n_frozen']:,}  clusters={primary['frozen_clusters']:,}")
print(f"  identified set [{primary['beta_min']:+.3f}, {primary['beta_max']:+.3f}]  "
      f"width {primary['width']:.3f} vs CI width {primary['ci_width']:.3f}")
print("  the set carries no sampling error and is never combined with the CI")

n_ambiguous_turns = multiple[list(ot.TURN_KEY)].drop_duplicates().shape[0]
print()
print(f"populations, stated rather than implied: the primary specification is fitted on "
      f"the {primary['n_frozen']:,} single-candidate turns; the variants on "
      f"{len(single):,} + {n_ambiguous_turns:,} = "
      f"{len(single) + n_ambiguous_turns:,} turns")

In [ ]:
# §3 Route A — the no-tracking route, only on turns the frozen rule did not take
frame_all = aa.all_candidates_frame(TURNS, PLAYER_PLAY)
frozen_keys = list(map(tuple, single[list(ot.TURN_KEY)].drop_duplicates()
                       .to_numpy().tolist()))
route_eligible = (frame_all[list(ot.TURN_KEY)].drop_duplicates().shape[0]
                  - len(frozen_keys))

route_turns, route_audit = aa.route_a_tables(frame_all, PLAYER_PLAY,
                                             exclude=frozen_keys,
                                             players_path=PLAYERS)

print(f"eligible turns: {route_eligible:,} of "
      f"{frame_all[list(ot.TURN_KEY)].drop_duplicates().shape[0]:,} with a visible "
      f"opposing route-runner, after the {len(frozen_keys):,} frozen ones")
print(route_audit.to_string(index=False))
print()
print(f"Route A attributed {len(route_turns):,} of {route_eligible:,} "
      f"({(len(route_turns) / route_eligible if route_eligible else 0):.1%})")
if len(route_turns):
    print(route_turns["attribution_label"].value_counts().to_string())
    print("coverage label of those turns: "
          + " · ".join(f"{k} {v:,}" for k, v in
                       route_turns["coverage_family"].fillna("MISSING")
                       .value_counts().items()))

In [ ]:
# §4 geometric slots — 5 yd then 8 yd, on what no earlier slot took
geo = aa.geometry_slots(TURNS, PLAYER_PLAY, frame_all,
                        tracking=(TRACKING or None),
                        exclude=frozen_keys, radii=RADII)

print(f"status: {geo['status']}")
print(f"turns considered {geo['n_turns_considered']:,}  ·  "
      f"N_geo({RADII[0]:g}) {geo['n_geo_5']:,}  ·  N_geo({RADII[1]:g}) {geo['n_geo_8']:,}")
print(f"paths tried: {geo['paths_tried']}")
if len(geo["rows"]):
    print("per-turn status: "
          + " · ".join(f"{k} {v:,}" for k, v in
                       geo["rows"]["geometry_status"].value_counts().items()))

# What the tracking file is, as a finding — never a reason to stop (AC8).
if TRACKING:
    import duckdb

    paths = []
    for raw in TRACKING:
        p = Path(raw).expanduser()
        if any(ch in p.name for ch in "*?["):
            paths.extend(sorted(str(q) for q in p.parent.glob(p.name)))
        else:
            paths.append(str(p))
    paths = [Path(p).resolve().as_posix() for p in paths if Path(p).exists()]
    if paths:
        con = duckdb.connect()
        try:
            row = con.execute(
                "SELECT COUNT(*) AS rows, "
                "COUNT(DISTINCT game_id || '-' || play_id) AS plays, "
                "COUNT(DISTINCT nfl_id) AS players, "
                "COUNT(*) * 1.0 / COUNT(DISTINCT game_id || '-' || play_id) AS rows_per_play "
                f"FROM read_csv({paths!r}, header=true, all_varchar=true)").fetchone()
        finally:
            con.close()
        print()
        print(f"tracking file(s) {len(paths)}: {int(row[0]):,} rows · "
              f"{int(row[1]):,} plays · {int(row[2]):,} players · "
              f"{float(row[3]):.2f} rows per play")
        if int(row[0]) == 1_048_575:
            print("  FINDING: 1,048,575 rows is a truncation, not a season — the "
                  "geometric slots are a code-path check here, not a result.")

In [ ]:
# The frozen 3-yd step on the SINGLE-candidate set — a check, not a recovery route
singles = aa.single_candidate_geometry(TURNS, single, tracking=(TRACKING or None))

print(f"single-candidate turns : {singles['n_turns']:,}")
print(f"status                 : {singles['status']}")
print(f"tracked                : {singles['n_tracked']:,} of {singles['n_turns']:,}"
      f" ({singles['n_tracked'] / singles['n_turns']:.1%})")
print(f"within {singles['radius_yd']:g} yd at the break frame: "
      f"{singles['n_within_radius']:,} of {singles['n_tracked']:,} tracked turns")
print(f"  D204 reported {singles['d204_reported']:,} on the full file from the laptop; "
      f"this run counts its own number and does not assume that one")
if len(singles["rows"]):
    print("per-turn status: "
          + " · ".join(f"{k} {v:,}" for k, v in
                       singles["rows"]["geometry_status"].value_counts().items()))
print()
print("These turns are already inside N_frozen: this is the frozen set's own 3-yd "
      "test, it adds nothing to N, and it is not a recovery route.")


In [ ]:
# §0 — the one number, with its composition underneath
by_rule = (route_turns["route_a_rule"].value_counts().to_dict()
           if len(route_turns) else {})
coverage = (route_turns["coverage_family"].fillna("MISSING").value_counts().to_dict()
            if len(route_turns) else {})

assembled = aa.assemble_attribution({
    "n_frozen": int(len(single)),
    "n_geo_5": int(geo["n_geo_5"]),
    "n_geo_8": int(geo["n_geo_8"]),
    "n_route_a": int(len(route_turns)),
    "band_turns": band_turns,
    "status": {"geo_5": geo["status"], "geo_8": geo["status"]},
    "route_a_by_rule": by_rule,
    "coverage": coverage,
})

print(f"N = {assembled['N']:,} of {band_turns:,} band turns "
      f"({assembled['share_of_band']:.1%})")
print()
print(pd.DataFrame(assembled["composition"])[
    ["slot", "label", "n", "share_of_N", "share_of_band"]]
      .to_string(index=False, float_format=lambda v: f"{v:.3f}"))
print()
print(f"RULE share of N      {assembled['rule_share_of_N']:.1%}")
print(f"INFERENCE share of N {assembled['inference_share_of_N']:.1%}")
print(f"the frozen slot is the {len(single):,} single-candidate turns (RULE, no "
      f"disambiguation needed); the {len(route_turns):,} recovered turns are ones it "
      f"did not take, each labelled INFERENCE")
if by_rule:
    print("Route A by rule: " + " · ".join(f"{k} {v:,}" for k, v in by_rule.items()))
if coverage:
    print("coverage label: " + " · ".join(f"{k} {v:,}" for k, v in coverage.items()))

agree = aa.agreement(route_turns, geo["rows"])
if len(agree) and int(agree.loc[agree["scope"] == "overall", "n_both"].sum()):
    print()
    print("Cross-check, turns where both routes fired:")
    print(agree.to_string(index=False, float_format=lambda v: f"{v:.3f}"))
else:
    print()
    print("Cross-check where both routes fired: NOT EVALUABLE — no tracking on "
          "this machine, so no turn was paired by both routes.")

if WRITE_ARTEFACTS:
    OUT.mkdir(parents=True, exist_ok=True)
    route_turns.to_csv(OUT / "route_a_turns.csv", index=False, encoding="utf-8")
    route_audit.to_csv(OUT / "route_a_audit.csv", index=False, encoding="utf-8")
    pd.DataFrame(assembled["composition"]).to_csv(
        OUT / "assembly.csv", index=False, encoding="utf-8")
    print(f"\nwrote 3 files to {OUT}")

In [ ]:
# AC10 — the two-receiver anchor, recomputed from player_play.csv
align = aa.alignment_counts(PLAYER_PLAY)
print(align[["pair", "mixed", "n_plays", "all_plays", "spec_anchor", "matches"]]
      .to_string(index=False))
print()
print("'matches' compares this recomputation with the anchor the spec quotes; "
      "a False is printed, never adjusted away.")

### What this cannot say

1. **Which man he covered.** No field identifies the opponent; every slot except
   `N_frozen` is an `INFERENCE`.
2. **That `N` is a coverage count.** It is a count of attributions with stated
   evidence; the `RULE` and `INFERENCE` shares are printed beside it.
3. **Anything about the turns where the cohort hides candidates** — they are
   outside this cycle and outside every denominator here.
4. **That the geometric slots are free.** With no full tracking they are zero
   with `NOT EVALUABLE`; with tracking, the agreement rate printed above is the
   error rate to quote for the no-tracking turns.
5. `two_candidate_separation_gap_yd` is MAX(sep) − MIN(sep) of the candidates'
   separation values, not a distance between two players.